<a href="https://colab.research.google.com/github/AlvaFM/odontoverse/blob/main/ODONTO_AI_v3_Ablation_Augmentation_Checkpoints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ODONTO-AI v3 — YOLO fine-tuning, data augmentation, checkpoints y ablation `restored`

Este cuaderno ejecuta dos experimentos comparables:

| Experimento | Clases | Objetivo |
|---|---|---|
| **A. with_restored** | `restored`, `caries`, `endodontic_or_post`, `prosthetic_or_artificial`, `third_molar_or_impacted`, `incisal_wear`, `structural_loss` | Modelo principal con restauraciones |
| **B. no_restored** | `caries`, `endodontic_or_post`, `prosthetic_or_artificial`, `third_molar_or_impacted`, `incisal_wear`, `structural_loss` | Ablation para evaluar si `restored` domina el aprendizaje |

Cambios incluidos:

1. Transfer learning / fine-tuning desde `yolo11s.pt`.
2. Data augmentation moderada.
3. `epochs: 150`.
4. Early stopping con `patience: 10`.
5. `save_period: 5` para checkpoints.
6. Conservación de imágenes negativas con `.txt` vacío.
7. Guardado de `best.pt` y `last.pt` por experimento.
8. Comparación final de métricas entre ambos experimentos.

In [1]:
# =========================
# 1. Montar Google Drive
# =========================
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# =========================
# 2. Configuración general
# =========================
from pathlib import Path
import shutil
import os

REPO_DIR = Path("/content/odonto-ai")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/odonto-ai")

# Dataset esperado en Drive:
# /content/drive/MyDrive/odonto-ai/data/raw/images.zip
# /content/drive/MyDrive/odonto-ai/data/raw/annotations/

DRIVE_RAW_DIR = DRIVE_PROJECT_DIR / "data" / "raw"
DRIVE_IMAGES_ZIP = DRIVE_RAW_DIR / "images.zip"
DRIVE_IMAGES_DIR = DRIVE_RAW_DIR / "images"
DRIVE_ANNOTATIONS_DIR = DRIVE_RAW_DIR / "annotations"

RUNS_DIR = DRIVE_PROJECT_DIR / "runs"
MODELS_DIR = DRIVE_PROJECT_DIR / "models"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_PROJECT_DIR:", DRIVE_PROJECT_DIR)
print("DRIVE_RAW_DIR:", DRIVE_RAW_DIR)
print("DRIVE_IMAGES_ZIP existe:", DRIVE_IMAGES_ZIP.exists())
print("DRIVE_IMAGES_DIR existe:", DRIVE_IMAGES_DIR.exists())
print("DRIVE_ANNOTATIONS_DIR existe:", DRIVE_ANNOTATIONS_DIR.exists())

DRIVE_PROJECT_DIR: /content/drive/MyDrive/odonto-ai
DRIVE_RAW_DIR: /content/drive/MyDrive/odonto-ai/data/raw
DRIVE_IMAGES_ZIP existe: True
DRIVE_IMAGES_DIR existe: False
DRIVE_ANNOTATIONS_DIR existe: True


In [3]:
# =========================
# 3. Crear proyecto limpio en /content/odonto-ai
# =========================
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

for folder in [
    "configs",
    "data/raw",
    "data/processed",
    "data/reports",
    "models",
    "src/dataset",
    "src/training",
    "src/inference",
    "src/api",
    "tests",
]:
    (REPO_DIR / folder).mkdir(parents=True, exist_ok=True)

# Archivos __init__.py
for init_file in [
    "src/__init__.py",
    "src/dataset/__init__.py",
    "src/training/__init__.py",
    "src/inference/__init__.py",
    "src/api/__init__.py",
    "tests/__init__.py",
]:
    (REPO_DIR / init_file).write_text("", encoding="utf-8")

%cd /content/odonto-ai
!find . -maxdepth 3 -type d | sort

/content/odonto-ai
.
./configs
./data
./data/processed
./data/raw
./data/reports
./models
./src
./src/api
./src/dataset
./src/inference
./src/training
./tests


In [4]:
# =========================
# 4. Instalar dependencias
# =========================
!pip install -q ultralytics pyyaml pillow scikit-learn pandas numpy opencv-python matplotlib

import torch
print("torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.6 MB/s eta 0:00:00
torch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [5]:
# =========================
# 5. Copiar dataset desde Drive
# =========================
RAW_DIR = REPO_DIR / "data" / "raw"
IMAGES_DIR = RAW_DIR / "images"
ANNOTATIONS_DIR = RAW_DIR / "annotations"

RAW_DIR.mkdir(parents=True, exist_ok=True)

# Copiar annotations/
if ANNOTATIONS_DIR.exists():
    shutil.rmtree(ANNOTATIONS_DIR)

if not DRIVE_ANNOTATIONS_DIR.exists():
    raise FileNotFoundError(f"No existe la carpeta de anotaciones en Drive: {DRIVE_ANNOTATIONS_DIR}")

shutil.copytree(DRIVE_ANNOTATIONS_DIR, ANNOTATIONS_DIR)
print("Annotations copiadas.")

# Copiar/descomprimir imágenes
if IMAGES_DIR.exists():
    shutil.rmtree(IMAGES_DIR)

IMAGES_DIR.mkdir(parents=True, exist_ok=True)

if DRIVE_IMAGES_ZIP.exists():
    shutil.copy2(DRIVE_IMAGES_ZIP, RAW_DIR / "images.zip")
    !unzip -q -o /content/odonto-ai/data/raw/images.zip -d /content/odonto-ai/data/raw/images
elif DRIVE_IMAGES_DIR.exists():
    shutil.copytree(DRIVE_IMAGES_DIR, IMAGES_DIR, dirs_exist_ok=True)
else:
    raise FileNotFoundError(
        "No se encontró images.zip ni images/ en Drive. "
        "Debe existir /content/drive/MyDrive/odonto-ai/data/raw/images.zip "
        "o /content/drive/MyDrive/odonto-ai/data/raw/images/"
    )

# Corregir caso images/images/
nested = IMAGES_DIR / "images"
if nested.exists():
    for file_path in nested.iterdir():
        shutil.move(str(file_path), str(IMAGES_DIR / file_path.name))
    shutil.rmtree(nested)

image_count = len(list(IMAGES_DIR.glob("*.jpg"))) + len(list(IMAGES_DIR.glob("*.jpeg"))) + len(list(IMAGES_DIR.glob("*.png")))
print("Imágenes:", image_count)
print("Archivos en annotations:")
!ls -lah /content/odonto-ai/data/raw/annotations | head -30
!ls -lah /content/odonto-ai/data/raw/images | head -10

Annotations copiadas.
Imágenes: 924
Archivos en annotations:
total 15M
drwx------ 2 root root 4.0K Jun  8 14:57 .
drwxr-xr-x 4 root root 4.0K Jun 20 05:04 ..
-rw------- 1 root root 5.3M Apr 27 05:59 mouth_and_teeth_labels.json
-rw------- 1 root root 9.0M Apr 27 05:59 teeth_fdi_labels.json
total 943M
drwxr-xr-x 2 root root   40K Jun 20 05:05 .
drwxr-xr-x 4 root root  4.0K Jun 20 05:04 ..
-rw-rw-rw- 1 root root  1.1M Apr 27 06:00 1000-F-19.jpg
-rw-rw-rw- 1 root root  1.1M Apr 27 06:00 1001-F-20.jpg
-rw-rw-rw- 1 root root  1.1M Apr 27 06:00 1002-F-20.jpg
-rw-rw-rw- 1 root root  996K Apr 27 06:00 1003-F-23.jpg
-rw-rw-rw- 1 root root  965K Apr 27 06:00 1005-F-19.jpg
-rw-rw-rw- 1 root root  1.1M Apr 27 06:00 1006-F-25.jpg
-rw-rw-rw- 1 root root  858K Apr 27 06:00 1007-M-28.jpg


In [6]:
# =========================
# 6. Escribir scripts src/
# =========================
from pathlib import Path

files = {
    "src/dataset/audit.py": '\nfrom __future__ import annotations\n\nimport csv\nimport json\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Any\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nIMAGES_DIR = PROJECT_ROOT / "data" / "raw" / "images"\nANNOTATIONS_DIR = PROJECT_ROOT / "data" / "raw" / "annotations"\nCOCO_PATH = ANNOTATIONS_DIR / "mouth_and_teeth_labels.json"\n\nREPORTS_DIR = PROJECT_ROOT / "data" / "reports"\nSUMMARY_PATH = REPORTS_DIR / "dataset_summary.json"\nCLASS_COUNTS_PATH = REPORTS_DIR / "class_counts.csv"\n\nIMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}\n\nMOUTH_CLASSES = {"Ed", "De", "Me", "Mne"}\nBASE_IGNORE_CLASSES = {"Ed", "De", "Me", "Mne", "H"}\n\nCLASS_GROUPS_WITH_RESTORED = {\n    "R": "restored",\n    "C": "caries",\n    "Te": "endodontic_or_post",\n    "TeM": "endodontic_or_post",\n    "Ri": "endodontic_or_post",\n    "RiM": "endodontic_or_post",\n    "CpuM": "prosthetic_or_artificial",\n    "P": "prosthetic_or_artificial",\n    "Im": "prosthetic_or_artificial",\n    "M3i": "third_molar_or_impacted",\n    "M3f": "third_molar_or_impacted",\n    "I": "third_molar_or_impacted",\n    "Di": "incisal_wear",\n    "Rr": "structural_loss",\n    "Dc": "structural_loss",\n}\n\nCLASS_GROUPS_NO_RESTORED = {\n    key: value\n    for key, value in CLASS_GROUPS_WITH_RESTORED.items()\n    if key != "R"\n}\n\n\ndef load_json(path: Path) -> dict[str, Any]:\n    if not path.exists():\n        raise FileNotFoundError(f"No existe el archivo requerido: {path}")\n\n    with path.open("r", encoding="utf-8") as file:\n        return json.load(file)\n\n\ndef list_image_files() -> list[Path]:\n    if not IMAGES_DIR.exists():\n        raise FileNotFoundError(f"No existe la carpeta de imágenes: {IMAGES_DIR}")\n\n    return sorted(\n        path\n        for path in IMAGES_DIR.iterdir()\n        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS\n    )\n\n\ndef has_valid_bbox(annotation: dict[str, Any]) -> bool:\n    bbox = annotation.get("bbox")\n    if not isinstance(bbox, list) or len(bbox) != 4:\n        return False\n    _, _, width, height = bbox\n    return width > 0 and height > 0\n\n\ndef has_valid_segmentation(annotation: dict[str, Any]) -> bool:\n    segmentation = annotation.get("segmentation")\n    if not isinstance(segmentation, list) or not segmentation:\n        return False\n    first_polygon = segmentation[0]\n    return isinstance(first_polygon, list) and len(first_polygon) >= 6\n\n\ndef write_counter_csv(counter: Counter[str], path: Path) -> None:\n    with path.open("w", encoding="utf-8", newline="") as file:\n        writer = csv.writer(file)\n        writer.writerow(["class_name", "count"])\n        for class_name, count in counter.most_common():\n            writer.writerow([class_name, count])\n\n\ndef audit_dataset() -> None:\n    REPORTS_DIR.mkdir(parents=True, exist_ok=True)\n\n    image_files = list_image_files()\n    coco = load_json(COCO_PATH)\n\n    coco_images = coco.get("images", [])\n    coco_annotations = coco.get("annotations", [])\n    coco_categories = coco.get("categories", [])\n\n    image_id_to_file_name = {\n        image["id"]: image["file_name"]\n        for image in coco_images\n        if "id" in image and "file_name" in image\n    }\n\n    category_id_to_name = {\n        category["id"]: category["name"]\n        for category in coco_categories\n        if "id" in category and "name" in category\n    }\n\n    disk_image_names = {path.name for path in image_files}\n    coco_image_names = set(image_id_to_file_name.values())\n\n    raw_class_counter: Counter[str] = Counter()\n    grouped_with_restored_counter: Counter[str] = Counter()\n    grouped_no_restored_counter: Counter[str] = Counter()\n\n    invalid_image_id_count = 0\n    missing_geometry_count = 0\n\n    for annotation in coco_annotations:\n        image_id = annotation.get("image_id")\n        category_id = annotation.get("category_id")\n\n        if image_id not in image_id_to_file_name:\n            invalid_image_id_count += 1\n\n        class_name = category_id_to_name.get(category_id, f"unknown_{category_id}")\n        raw_class_counter[class_name] += 1\n\n        if not (has_valid_bbox(annotation) or has_valid_segmentation(annotation)):\n            missing_geometry_count += 1\n\n        if class_name in CLASS_GROUPS_WITH_RESTORED:\n            grouped_with_restored_counter[CLASS_GROUPS_WITH_RESTORED[class_name]] += 1\n\n        if class_name in CLASS_GROUPS_NO_RESTORED:\n            grouped_no_restored_counter[CLASS_GROUPS_NO_RESTORED[class_name]] += 1\n\n    summary = {\n        "total_images_on_disk": len(image_files),\n        "total_images_in_coco": len(coco_images),\n        "total_annotations": len(coco_annotations),\n        "total_categories": len(coco_categories),\n        "images_without_annotation_record": sorted(disk_image_names - coco_image_names),\n        "annotation_records_without_image": sorted(coco_image_names - disk_image_names),\n        "invalid_image_id_annotations": invalid_image_id_count,\n        "annotations_without_bbox_or_segmentation": missing_geometry_count,\n        "raw_class_counts": dict(raw_class_counter),\n        "grouped_with_restored_counts": dict(grouped_with_restored_counter),\n        "grouped_no_restored_counts": dict(grouped_no_restored_counter),\n    }\n\n    with SUMMARY_PATH.open("w", encoding="utf-8") as file:\n        json.dump(summary, file, indent=2, ensure_ascii=False)\n\n    write_counter_csv(raw_class_counter, CLASS_COUNTS_PATH)\n    write_counter_csv(grouped_with_restored_counter, REPORTS_DIR / "grouped_with_restored_counts.csv")\n    write_counter_csv(grouped_no_restored_counter, REPORTS_DIR / "grouped_no_restored_counts.csv")\n\n    print("Auditoría finalizada.")\n    print(json.dumps({\n        "total_images_on_disk": summary["total_images_on_disk"],\n        "total_images_in_coco": summary["total_images_in_coco"],\n        "total_annotations": summary["total_annotations"],\n        "total_categories": summary["total_categories"],\n        "invalid_image_id_annotations": summary["invalid_image_id_annotations"],\n        "annotations_without_bbox_or_segmentation": summary["annotations_without_bbox_or_segmentation"],\n        "grouped_with_restored_counts": summary["grouped_with_restored_counts"],\n        "grouped_no_restored_counts": summary["grouped_no_restored_counts"],\n    }, indent=2, ensure_ascii=False))\n    print(f"Reporte JSON: {SUMMARY_PATH}")\n\n\nif __name__ == "__main__":\n    audit_dataset()\n',
    "src/dataset/split_dataset.py": '\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nfrom sklearn.model_selection import train_test_split\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nIMAGES_DIR = PROJECT_ROOT / "data" / "raw" / "images"\nANNOTATIONS_DIR = PROJECT_ROOT / "data" / "raw" / "annotations"\nCOCO_PATH = ANNOTATIONS_DIR / "mouth_and_teeth_labels.json"\nSPLITS_DIR = PROJECT_ROOT / "data" / "processed" / "splits"\n\nRANDOM_STATE = 42\nTRAIN_SIZE = 0.70\nVAL_SIZE = 0.15\nTEST_SIZE = 0.15\n\n\ndef load_coco_images() -> list[str]:\n    if not COCO_PATH.exists():\n        raise FileNotFoundError(f"No existe el archivo: {COCO_PATH}")\n\n    with COCO_PATH.open("r", encoding="utf-8") as file:\n        coco = json.load(file)\n\n    image_names = [\n        image["file_name"]\n        for image in coco.get("images", [])\n        if (IMAGES_DIR / image["file_name"]).exists()\n    ]\n\n    if not image_names:\n        raise ValueError("No se encontraron imágenes válidas para dividir.")\n\n    return sorted(image_names)\n\n\ndef save_split(name: str, image_names: list[str]) -> None:\n    SPLITS_DIR.mkdir(parents=True, exist_ok=True)\n    path = SPLITS_DIR / f"{name}.txt"\n    path.write_text("\\n".join(image_names), encoding="utf-8")\n\n\ndef validate_no_overlap(train: list[str], val: list[str], test: list[str]) -> None:\n    train_set, val_set, test_set = set(train), set(val), set(test)\n    if train_set & val_set:\n        raise ValueError("Hay overlap entre train y val.")\n    if train_set & test_set:\n        raise ValueError("Hay overlap entre train y test.")\n    if val_set & test_set:\n        raise ValueError("Hay overlap entre val y test.")\n\n\ndef split_dataset() -> None:\n    if abs((TRAIN_SIZE + VAL_SIZE + TEST_SIZE) - 1.0) > 1e-9:\n        raise ValueError("TRAIN_SIZE + VAL_SIZE + TEST_SIZE debe sumar 1.0")\n\n    image_names = load_coco_images()\n\n    train_names, temp_names = train_test_split(\n        image_names,\n        test_size=(1.0 - TRAIN_SIZE),\n        random_state=RANDOM_STATE,\n        shuffle=True,\n    )\n\n    relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)\n\n    val_names, test_names = train_test_split(\n        temp_names,\n        test_size=relative_test_size,\n        random_state=RANDOM_STATE,\n        shuffle=True,\n    )\n\n    train_names = sorted(train_names)\n    val_names = sorted(val_names)\n    test_names = sorted(test_names)\n\n    validate_no_overlap(train_names, val_names, test_names)\n\n    save_split("train", train_names)\n    save_split("val", val_names)\n    save_split("test", test_names)\n\n    print("Split finalizado.")\n    print(f"Train: {len(train_names)}")\n    print(f"Validation: {len(val_names)}")\n    print(f"Test: {len(test_names)}")\n\n\nif __name__ == "__main__":\n    split_dataset()\n',
    "src/dataset/convert_coco_to_yolo.py": '\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nfrom pathlib import Path\nfrom typing import Any\n\nfrom PIL import Image\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nIMAGES_DIR = PROJECT_ROOT / "data" / "raw" / "images"\nANNOTATIONS_DIR = PROJECT_ROOT / "data" / "raw" / "annotations"\nCOCO_PATH = ANNOTATIONS_DIR / "mouth_and_teeth_labels.json"\nSPLITS_DIR = PROJECT_ROOT / "data" / "processed" / "splits"\n\nBASE_IGNORE_CLASSES = {"Ed", "De", "Me", "Mne", "H"}\n\nCLASS_GROUPS_BASE = {\n    "R": "restored",\n    "C": "caries",\n    "Te": "endodontic_or_post",\n    "TeM": "endodontic_or_post",\n    "Ri": "endodontic_or_post",\n    "RiM": "endodontic_or_post",\n    "CpuM": "prosthetic_or_artificial",\n    "P": "prosthetic_or_artificial",\n    "Im": "prosthetic_or_artificial",\n    "M3i": "third_molar_or_impacted",\n    "M3f": "third_molar_or_impacted",\n    "I": "third_molar_or_impacted",\n    "Di": "incisal_wear",\n    "Rr": "structural_loss",\n    "Dc": "structural_loss",\n}\n\n\ndef get_class_config(variant: str) -> tuple[set[str], dict[str, str], dict[str, int]]:\n    if variant == "with_restored":\n        ignore_classes = set(BASE_IGNORE_CLASSES)\n        class_groups = dict(CLASS_GROUPS_BASE)\n        yolo_classes = {\n            "restored": 0,\n            "caries": 1,\n            "endodontic_or_post": 2,\n            "prosthetic_or_artificial": 3,\n            "third_molar_or_impacted": 4,\n            "incisal_wear": 5,\n            "structural_loss": 6,\n        }\n        return ignore_classes, class_groups, yolo_classes\n\n    if variant == "no_restored":\n        ignore_classes = set(BASE_IGNORE_CLASSES) | {"R"}\n        class_groups = {\n            key: value\n            for key, value in CLASS_GROUPS_BASE.items()\n            if key != "R"\n        }\n        yolo_classes = {\n            "caries": 0,\n            "endodontic_or_post": 1,\n            "prosthetic_or_artificial": 2,\n            "third_molar_or_impacted": 3,\n            "incisal_wear": 4,\n            "structural_loss": 5,\n        }\n        return ignore_classes, class_groups, yolo_classes\n\n    raise ValueError(f"Variante no soportada: {variant}")\n\n\ndef load_json(path: Path) -> dict[str, Any]:\n    if not path.exists():\n        raise FileNotFoundError(f"No existe el archivo: {path}")\n    with path.open("r", encoding="utf-8") as file:\n        return json.load(file)\n\n\ndef load_split(split_name: str) -> set[str]:\n    split_path = SPLITS_DIR / f"{split_name}.txt"\n    if not split_path.exists():\n        raise FileNotFoundError(\n            f"No existe {split_path}. Ejecuta primero src/dataset/split_dataset.py"\n        )\n    return {\n        line.strip()\n        for line in split_path.read_text(encoding="utf-8").splitlines()\n        if line.strip()\n    }\n\n\ndef extract_bbox(annotation: dict[str, Any]) -> list[float] | None:\n    bbox = annotation.get("bbox")\n    if isinstance(bbox, list) and len(bbox) == 4:\n        x_min, y_min, width, height = bbox\n        if width > 0 and height > 0:\n            return [float(x_min), float(y_min), float(width), float(height)]\n\n    segmentation = annotation.get("segmentation")\n    if not segmentation:\n        return None\n\n    polygon = segmentation[0]\n    if not polygon or len(polygon) < 6:\n        return None\n\n    x_points = polygon[0::2]\n    y_points = polygon[1::2]\n\n    x_min = min(x_points)\n    y_min = min(y_points)\n    x_max = max(x_points)\n    y_max = max(y_points)\n\n    width = x_max - x_min\n    height = y_max - y_min\n\n    if width <= 0 or height <= 0:\n        return None\n\n    return [float(x_min), float(y_min), float(width), float(height)]\n\n\ndef coco_bbox_to_yolo(\n    bbox: list[float],\n    image_width: int,\n    image_height: int,\n) -> tuple[float, float, float, float] | None:\n    x_min, y_min, width, height = bbox\n\n    x_center = (x_min + width / 2) / image_width\n    y_center = (y_min + height / 2) / image_height\n    width_norm = width / image_width\n    height_norm = height / image_height\n\n    values = (x_center, y_center, width_norm, height_norm)\n\n    if not all(0 <= value <= 1 for value in values):\n        return None\n\n    if width_norm <= 0 or height_norm <= 0:\n        return None\n\n    return values\n\n\ndef prepare_output_dirs(output_dir: Path) -> None:\n    if output_dir.exists():\n        shutil.rmtree(output_dir)\n\n    for split_name in ["train", "val", "test"]:\n        (output_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)\n        (output_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)\n\n\ndef build_image_annotation_map(coco: dict[str, Any]) -> dict[str, list[dict[str, Any]]]:\n    image_id_to_file_name = {\n        image["id"]: image["file_name"]\n        for image in coco.get("images", [])\n    }\n\n    annotations_by_file_name: dict[str, list[dict[str, Any]]] = {}\n\n    for annotation in coco.get("annotations", []):\n        image_id = annotation.get("image_id")\n        file_name = image_id_to_file_name.get(image_id)\n        if file_name is None:\n            continue\n        annotations_by_file_name.setdefault(file_name, []).append(annotation)\n\n    return annotations_by_file_name\n\n\ndef write_data_yaml(output_dir: Path, yolo_classes: dict[str, int]) -> None:\n    class_names_by_id = {\n        class_id: class_name\n        for class_name, class_id in yolo_classes.items()\n    }\n\n    names_text = "\\n".join(\n        f"  {class_id}: {class_names_by_id[class_id]}"\n        for class_id in sorted(class_names_by_id)\n    )\n\n    yaml_text = f"""path: {output_dir.relative_to(PROJECT_ROOT).as_posix()}\n\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n{names_text}\n"""\n    (output_dir / "data.yaml").write_text(yaml_text, encoding="utf-8")\n\n\ndef convert_coco_to_yolo(variant: str, output_dir: Path) -> None:\n    ignore_classes, class_groups, yolo_classes = get_class_config(variant)\n\n    coco = load_json(COCO_PATH)\n\n    categories = {\n        category["id"]: category["name"]\n        for category in coco.get("categories", [])\n    }\n\n    annotations_by_file_name = build_image_annotation_map(coco)\n\n    split_map = {\n        "train": load_split("train"),\n        "val": load_split("val"),\n        "test": load_split("test"),\n    }\n\n    prepare_output_dirs(output_dir)\n\n    converted_images = 0\n    converted_annotations = 0\n    skipped_annotations = 0\n    negative_images = 0\n\n    for split_name, file_names in split_map.items():\n        for file_name in sorted(file_names):\n            image_path = IMAGES_DIR / file_name\n            if not image_path.exists():\n                continue\n\n            with Image.open(image_path) as image:\n                image_width, image_height = image.size\n\n            yolo_lines: list[str] = []\n\n            for annotation in annotations_by_file_name.get(file_name, []):\n                original_class = categories.get(annotation.get("category_id"))\n\n                if original_class in ignore_classes:\n                    continue\n\n                grouped_class = class_groups.get(original_class)\n\n                if grouped_class is None:\n                    skipped_annotations += 1\n                    continue\n\n                bbox = extract_bbox(annotation)\n                if bbox is None:\n                    skipped_annotations += 1\n                    continue\n\n                yolo_bbox = coco_bbox_to_yolo(\n                    bbox=bbox,\n                    image_width=image_width,\n                    image_height=image_height,\n                )\n\n                if yolo_bbox is None:\n                    skipped_annotations += 1\n                    continue\n\n                class_id = yolo_classes[grouped_class]\n                x_center, y_center, width_norm, height_norm = yolo_bbox\n\n                yolo_lines.append(\n                    f"{class_id} {x_center:.6f} {y_center:.6f} "\n                    f"{width_norm:.6f} {height_norm:.6f}"\n                )\n\n            # Punto crítico:\n            # Se conservan imágenes sin hallazgos entrenables con .txt vacío.\n            # Esto crea ejemplos negativos y reduce falsos positivos.\n            shutil.copy2(image_path, output_dir / "images" / split_name / file_name)\n\n            label_path = output_dir / "labels" / split_name / f"{Path(file_name).stem}.txt"\n            label_path.write_text("\\n".join(yolo_lines), encoding="utf-8")\n\n            converted_images += 1\n            converted_annotations += len(yolo_lines)\n\n            if not yolo_lines:\n                negative_images += 1\n\n    write_data_yaml(output_dir, yolo_classes)\n\n    print("Conversión finalizada.")\n    print(f"Variante: {variant}")\n    print(f"Imágenes convertidas: {converted_images}")\n    print(f"Anotaciones convertidas: {converted_annotations}")\n    print(f"Imágenes negativas sin hallazgos entrenables: {negative_images}")\n    print(f"Anotaciones omitidas: {skipped_annotations}")\n    print(f"Salida: {output_dir}")\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\n        "--variant",\n        choices=["with_restored", "no_restored"],\n        required=True,\n        help="Variante de clases para generar dataset YOLO.",\n    )\n    parser.add_argument(\n        "--output",\n        required=True,\n        help="Directorio de salida del dataset YOLO procesado.",\n    )\n    return parser.parse_args()\n\n\nif __name__ == "__main__":\n    args = parse_args()\n    output_path = Path(args.output)\n    if not output_path.is_absolute():\n        output_path = PROJECT_ROOT / output_path\n    convert_coco_to_yolo(variant=args.variant, output_dir=output_path)\n',
    "src/training/train_yolo.py": '\nfrom __future__ import annotations\n\nimport shutil\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\nfrom ultralytics import YOLO\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nTRAIN_CONFIG_PATH = PROJECT_ROOT / "configs" / "train.yaml"\nLOCAL_MODELS_DIR = PROJECT_ROOT / "models"\n\nAUGMENTATION_KEYS = {\n    "hsv_h", "hsv_s", "hsv_v",\n    "degrees", "translate", "scale", "shear", "perspective",\n    "flipud", "fliplr", "mosaic", "mixup", "copy_paste",\n    "close_mosaic", "multi_scale",\n}\n\n\ndef resolve_path(path_value: str | Path) -> Path:\n    path = Path(path_value)\n    return path if path.is_absolute() else PROJECT_ROOT / path\n\n\ndef load_train_config() -> dict[str, Any]:\n    if not TRAIN_CONFIG_PATH.exists():\n        raise FileNotFoundError(f"No existe el archivo: {TRAIN_CONFIG_PATH}")\n\n    with TRAIN_CONFIG_PATH.open("r", encoding="utf-8") as file:\n        config = yaml.safe_load(file)\n\n    if not isinstance(config, dict):\n        raise ValueError("configs/train.yaml debe contener un diccionario YAML válido.")\n\n    return config\n\n\ndef train_yolo() -> None:\n    config = load_train_config()\n\n    model = YOLO(str(config["model"]))\n\n    train_kwargs: dict[str, Any] = {\n        "data": str(resolve_path(config["data"])),\n        "epochs": int(config["epochs"]),\n        "imgsz": int(config["imgsz"]),\n        "batch": int(config["batch"]),\n        "patience": int(config["patience"]),\n        "device": str(config["device"]),\n        "project": str(resolve_path(config["project"])),\n        "name": str(config["name"]),\n        "seed": int(config.get("seed", 42)),\n        "save_period": int(config.get("save_period", -1)),\n        "plots": bool(config.get("plots", True)),\n    }\n\n    for key in AUGMENTATION_KEYS:\n        if key in config:\n            train_kwargs[key] = config[key]\n\n    print("Configuración de entrenamiento:")\n    for key, value in train_kwargs.items():\n        print(f"  {key}: {value}")\n\n    results = model.train(**train_kwargs)\n\n    run_dir = Path(results.save_dir)\n    weights_dir = run_dir / "weights"\n    best_model_path = weights_dir / "best.pt"\n    last_model_path = weights_dir / "last.pt"\n\n    if not best_model_path.exists():\n        raise FileNotFoundError(f"No se encontró best.pt: {best_model_path}")\n\n    LOCAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)\n\n    experiment_name = str(config["name"])\n    local_best = LOCAL_MODELS_DIR / f"{experiment_name}_best.pt"\n    local_last = LOCAL_MODELS_DIR / f"{experiment_name}_last.pt"\n\n    shutil.copy2(best_model_path, local_best)\n    print(f"best.pt copiado a: {local_best}")\n\n    if last_model_path.exists():\n        shutil.copy2(last_model_path, local_last)\n        print(f"last.pt copiado a: {local_last}")\n\n    print("Entrenamiento finalizado.")\n    print(f"Run directory: {run_dir}")\n\n\nif __name__ == "__main__":\n    train_yolo()\n',
    "src/inference/predict.py": '\nfrom __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\nfrom PIL import Image\nfrom ultralytics import YOLO\n\n\nPROJECT_ROOT = Path(__file__).resolve().parents[2]\nINFERENCE_CONFIG_PATH = PROJECT_ROOT / "configs" / "inference.yaml"\nALLOWED_SUFFIXES = {".jpg", ".jpeg", ".png"}\n\n\ndef resolve_path(path_value: str | Path) -> Path:\n    path = Path(path_value)\n    return path if path.is_absolute() else PROJECT_ROOT / path\n\n\ndef load_inference_config() -> dict[str, Any]:\n    if not INFERENCE_CONFIG_PATH.exists():\n        raise FileNotFoundError(f"No existe el archivo: {INFERENCE_CONFIG_PATH}")\n\n    with INFERENCE_CONFIG_PATH.open("r", encoding="utf-8") as file:\n        config = yaml.safe_load(file)\n\n    if not isinstance(config, dict):\n        raise ValueError("configs/inference.yaml debe contener un diccionario YAML válido.")\n\n    return config\n\n\ndef load_model(model_path: str | Path) -> YOLO:\n    resolved_model_path = resolve_path(model_path)\n\n    if not resolved_model_path.exists():\n        raise FileNotFoundError(f"No existe el modelo: {resolved_model_path}")\n\n    return YOLO(str(resolved_model_path))\n\n\ndef serialize_results(model: YOLO, results: list[Any]) -> list[dict[str, Any]]:\n    detections: list[dict[str, Any]] = []\n\n    for result in results:\n        if result.boxes is None:\n            continue\n\n        for box in result.boxes:\n            class_id = int(box.cls[0])\n            confidence = float(box.conf[0])\n            x1, y1, x2, y2 = [float(value) for value in box.xyxy[0].tolist()]\n\n            detections.append(\n                {\n                    "class_id": class_id,\n                    "class_name": model.names[class_id],\n                    "confidence": confidence,\n                    "bbox": {\n                        "x1": x1,\n                        "y1": y1,\n                        "x2": x2,\n                        "y2": y2,\n                    },\n                }\n            )\n\n    return detections\n\n\ndef predict_pil_image(\n    model: YOLO,\n    image: Image.Image,\n    imgsz: int,\n    conf: float,\n    iou: float,\n    max_det: int,\n) -> list[dict[str, Any]]:\n    results = model.predict(\n        source=image,\n        imgsz=imgsz,\n        conf=conf,\n        iou=iou,\n        max_det=max_det,\n        verbose=False,\n    )\n    return serialize_results(model, results)\n\n\ndef predict_image_path(image_path: Path) -> dict[str, Any]:\n    image_path = resolve_path(image_path)\n\n    if not image_path.exists():\n        raise FileNotFoundError(f"No existe la imagen: {image_path}")\n\n    if image_path.suffix.lower() not in ALLOWED_SUFFIXES:\n        raise ValueError("La imagen debe estar en formato JPG, JPEG o PNG.")\n\n    config = load_inference_config()\n    model = load_model(config["model_path"])\n\n    image = Image.open(image_path).convert("RGB")\n\n    detections = predict_pil_image(\n        model=model,\n        image=image,\n        imgsz=int(config["imgsz"]),\n        conf=float(config["conf"]),\n        iou=float(config["iou"]),\n        max_det=int(config["max_det"]),\n    )\n\n    return {\n        "image": str(image_path),\n        "detections": detections,\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--image", required=True, help="Ruta de imagen JPG/JPEG/PNG.")\n    args = parser.parse_args()\n\n    output = predict_image_path(Path(args.image))\n    print(json.dumps(output, indent=2, ensure_ascii=False))\n\n\nif __name__ == "__main__":\n    main()\n',
}

for relative_path, content in files.items():
    path = REPO_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.strip() + "\n", encoding="utf-8")

!find src -maxdepth 3 -type f | sort

src/api/__init__.py
src/dataset/audit.py
src/dataset/convert_coco_to_yolo.py
src/dataset/__init__.py
src/dataset/split_dataset.py
src/inference/__init__.py
src/inference/predict.py
src/__init__.py
src/training/__init__.py
src/training/train_yolo.py


In [7]:
# =========================
# 7. Auditoría inicial y split
# =========================
%cd /content/odonto-ai

!python src/dataset/audit.py
!python src/dataset/split_dataset.py

/content/odonto-ai
Auditoría finalizada.
{
  "total_images_on_disk": 924,
  "total_images_in_coco": 924,
  "total_annotations": 20957,
  "total_categories": 20,
  "invalid_image_id_annotations": 0,
  "annotations_without_bbox_or_segmentation": 0,
  "grouped_with_restored_counts": {
    "third_molar_or_impacted": 496,
    "restored": 6481,
    "prosthetic_or_artificial": 601,
    "endodontic_or_post": 869,
    "incisal_wear": 312,
    "caries": 399,
    "structural_loss": 210
  },
  "grouped_no_restored_counts": {
    "third_molar_or_impacted": 496,
    "prosthetic_or_artificial": 601,
    "endodontic_or_post": 869,
    "incisal_wear": 312,
    "caries": 399,
    "structural_loss": 210
  }
}
Reporte JSON: /content/odonto-ai/data/reports/dataset_summary.json
Split finalizado.
Train: 646
Validation: 139
Test: 139


## Experimento A — con `restored`

Este es el modelo principal. Mantiene `restored` porque es una clase clínicamente útil y el modelo la aprende bien. No se eliminan cajas de forma parcial.

In [8]:
# =========================
# 8A. Convertir dataset YOLO con restored
# =========================
%cd /content/odonto-ai

!python src/dataset/convert_coco_to_yolo.py \
  --variant with_restored \
  --output data/processed/inredd_yolo_with_restored

!cat data/processed/inredd_yolo_with_restored/data.yaml

print("\nIDs de clase presentes en labels:")
!find data/processed/inredd_yolo_with_restored/labels -name "*.txt" -exec awk '{print $1}' {} \; | sort -n | uniq

/content/odonto-ai
Conversión finalizada.
Variante: with_restored
Imágenes convertidas: 924
Anotaciones convertidas: 9368
Imágenes negativas sin hallazgos entrenables: 135
Anotaciones omitidas: 0
Salida: /content/odonto-ai/data/processed/inredd_yolo_with_restored
path: data/processed/inredd_yolo_with_restored

train: images/train
val: images/val
test: images/test

names:
  0: restored
  1: caries
  2: endodontic_or_post
  3: prosthetic_or_artificial
  4: third_molar_or_impacted
  5: incisal_wear
  6: structural_loss

IDs de clase presentes en labels:
0
1
2
3
4
5
6


In [9]:
# =========================
# 9A. Configurar entrenamiento con restored
# =========================
%%writefile configs/train.yaml
model: yolo11s.pt
data: data/processed/inredd_yolo_with_restored/data.yaml
epochs: 150
imgsz: 1024
batch: 8
patience: 10
device: 0
project: /content/drive/MyDrive/odonto-ai/runs
name: expA_yolo11s_with_restored_aug150
seed: 42
save_period: 5
plots: true

# Data augmentation moderada
degrees: 5
translate: 0.05
scale: 0.25
shear: 2
perspective: 0.0005
fliplr: 0.0
flipud: 0.0
mosaic: 0.5
mixup: 0.05
copy_paste: 0.05
close_mosaic: 10
multi_scale: 0.15

Writing configs/train.yaml


In [10]:
# =========================
# 10A. Entrenar modelo A
# =========================
%cd /content/odonto-ai
!python src/training/train_yolo.py

/content/odonto-ai
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Configuración de entrenamiento:
  data: /content/odonto-ai/data/processed/inredd_yolo_with_restored/data.yaml
  epochs: 150
  imgsz: 1024
  batch: 8
  patience: 10
  device: 0
  project: /content/drive/MyDrive/odonto-ai/runs
  name: expA_yolo11s_with_restored_aug150
  seed: 42
  save_period: 5
  plots: True
  copy_paste: 0.05
  degrees: 5
  flipud: 0.0
  shear: 2
  mixup: 0.05
  fliplr: 0.0
  multi_scale: 0.15
  translate: 0.05
  scale: 0.25
  perspective: 0.0005
  close_mosaic: 10
  mosaic: 0.5
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=r

## Experimento B — sin `restored`

Este experimento evalúa si la clase `restored`, por tener muchos más datos, está dominando el aprendizaje. Aquí `R` se ignora completamente desde la conversión COCO→YOLO. No se borran cajas parcialmente dentro del mismo modelo.

In [11]:
# =========================
# 8B. Convertir dataset YOLO sin restored
# =========================
%cd /content/odonto-ai

!python src/dataset/convert_coco_to_yolo.py \
  --variant no_restored \
  --output data/processed/inredd_yolo_no_restored

!cat data/processed/inredd_yolo_no_restored/data.yaml

print("\nIDs de clase presentes en labels:")
!find data/processed/inredd_yolo_no_restored/labels -name "*.txt" -exec awk '{print $1}' {} \; | sort -n | uniq

/content/odonto-ai
Conversión finalizada.
Variante: no_restored
Imágenes convertidas: 924
Anotaciones convertidas: 2887
Imágenes negativas sin hallazgos entrenables: 249
Anotaciones omitidas: 0
Salida: /content/odonto-ai/data/processed/inredd_yolo_no_restored
path: data/processed/inredd_yolo_no_restored

train: images/train
val: images/val
test: images/test

names:
  0: caries
  1: endodontic_or_post
  2: prosthetic_or_artificial
  3: third_molar_or_impacted
  4: incisal_wear
  5: structural_loss

IDs de clase presentes en labels:
0
1
2
3
4
5


In [12]:
# =========================
# 9B. Configurar entrenamiento sin restored
# =========================
%%writefile configs/train.yaml
model: yolo11s.pt
data: data/processed/inredd_yolo_no_restored/data.yaml
epochs: 150
imgsz: 1024
batch: 8
patience: 10
device: 0
project: /content/drive/MyDrive/odonto-ai/runs
name: expB_yolo11s_no_restored_aug150
seed: 42
save_period: 5
plots: true

# Data augmentation moderada
degrees: 5
translate: 0.05
scale: 0.25
shear: 2
perspective: 0.0005
fliplr: 0.0
flipud: 0.0
mosaic: 0.5
mixup: 0.05
copy_paste: 0.05
close_mosaic: 10
multi_scale: 0.15

Overwriting configs/train.yaml


In [13]:
# =========================
# 10B. Entrenar modelo B
# =========================
%cd /content/odonto-ai
!python src/training/train_yolo.py

/content/odonto-ai
Configuración de entrenamiento:
  data: /content/odonto-ai/data/processed/inredd_yolo_no_restored/data.yaml
  epochs: 150
  imgsz: 1024
  batch: 8
  patience: 10
  device: 0
  project: /content/drive/MyDrive/odonto-ai/runs
  name: expB_yolo11s_no_restored_aug150
  seed: 42
  save_period: 5
  plots: True
  fliplr: 0.0
  shear: 2
  multi_scale: 0.15
  translate: 0.05
  perspective: 0.0005
  mosaic: 0.5
  degrees: 5
  scale: 0.25
  mixup: 0.05
  copy_paste: 0.05
  flipud: 0.0
  close_mosaic: 10
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.05, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/odonto-ai/data/processed/inredd_yolo_no_restored/data.yaml, degrees=5, deterministic=True, devic

## Comparación automática de resultados

Ejecuta esta celda cuando ambos entrenamientos hayan terminado. Compara el mejor valor de `mAP50`, `mAP50-95`, `precision` y `recall`.

In [14]:
# =========================
# 11. Comparar métricas A vs B
# =========================
from pathlib import Path
import pandas as pd

runs_root = Path("/content/drive/MyDrive/odonto-ai/runs")

experiments = {
    "A_with_restored": runs_root / "expA_yolo11s_with_restored_aug150" / "results.csv",
    "B_no_restored": runs_root / "expB_yolo11s_no_restored_aug150" / "results.csv",
}

rows = []

for name, csv_path in experiments.items():
    if not csv_path.exists():
        print(f"No existe results.csv para {name}: {csv_path}")
        continue

    df = pd.read_csv(csv_path)
    df.columns = [col.strip() for col in df.columns]

    metric_cols = {
        "precision": "metrics/precision(B)",
        "recall": "metrics/recall(B)",
        "mAP50": "metrics/mAP50(B)",
        "mAP50-95": "metrics/mAP50-95(B)",
    }

    best_idx = df[metric_cols["mAP50-95"]].idxmax()
    best_row = df.loc[best_idx]

    rows.append({
        "experiment": name,
        "best_epoch_by_mAP50-95": int(best_row["epoch"]) if "epoch" in best_row else int(best_idx),
        "precision": float(best_row[metric_cols["precision"]]),
        "recall": float(best_row[metric_cols["recall"]]),
        "mAP50": float(best_row[metric_cols["mAP50"]]),
        "mAP50-95": float(best_row[metric_cols["mAP50-95"]]),
        "results_csv": str(csv_path),
    })

comparison = pd.DataFrame(rows)
comparison

,experiment,best_epoch_by_mAP50-95,precision,recall,mAP50,mAP50-95,results_csv
0,A_with_restored,32,0.64625,0.66948,0.65589,0.36248,/content/drive/MyDrive/odonto-ai/runs/expA_yol...
1,B_no_restored,32,0.65129,0.57274,0.58483,0.30726,/content/drive/MyDrive/odonto-ai/runs/expB_yol...


## Reanudar entrenamiento si Colab se corta

Usa la celda correspondiente al experimento que quieres continuar.

In [ ]:
# =========================
# 12A. Reanudar experimento A
# =========================
from ultralytics import YOLO

last_ckpt_A = "/content/drive/MyDrive/odonto-ai/runs/expA_yolo11s_with_restored_aug150/weights/last.pt"

model = YOLO(last_ckpt_A)
model.train(resume=True)

WARNING ⚠️ model '/content/drive/MyDrive/odonto-ai/runs/expA_yolo11s_with_restored_aug150/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.72 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=No

In [ ]:
# =========================
# 12B. Reanudar experimento B
# =========================
from ultralytics import YOLO

last_ckpt_B = "/content/drive/MyDrive/odonto-ai/runs/expB_yolo11s_no_restored_aug150/weights/last.pt"

model = YOLO(last_ckpt_B)
model.train(resume=True)

## Uso posterior en la app

Para la app, usa el mejor modelo según el objetivo:

- Si buscas modelo general de hallazgos con restauraciones: `expA.../weights/best.pt`.
- Si buscas evaluar hallazgos sin que `restored` domine: `expB.../weights/best.pt`.

Copia el seleccionado a:

```text
/content/drive/MyDrive/odonto-ai/models/best.pt
```

In [17]:
# =========================
# 13. Copiar modelo elegido a models/best.pt
# Cambia SELECTED_EXPERIMENT según la decisión final.
# =========================
from pathlib import Path
import shutil

SELECTED_EXPERIMENT = "expA_yolo11s_with_restored_aug150"
# SELECTED_EXPERIMENT = "expB_yolo11s_no_restored_aug150"

source_best = Path("/content/drive/MyDrive/odonto-ai/runs") / SELECTED_EXPERIMENT / "weights" / "best.pt"
source_last = Path("/content/drive/MyDrive/odonto-ai/runs") / SELECTED_EXPERIMENT / "weights" / "last.pt"

target_dir = Path("/content/drive/MyDrive/odonto-ai/models")
target_dir.mkdir(parents=True, exist_ok=True)

if not source_best.exists():
    raise FileNotFoundError(f"No existe best.pt: {source_best}")

shutil.copy2(source_best, target_dir / "best.pt")

if source_last.exists():
    shutil.copy2(source_last, target_dir / "last.pt")

print("Modelo seleccionado:", SELECTED_EXPERIMENT)
print("best.pt:", target_dir / "best.pt")
print("last.pt:", target_dir / "last.pt")

Modelo seleccionado: expA_yolo11s_with_restored_aug150
best.pt: /content/drive/MyDrive/odonto-ai/models/best.pt
last.pt: /content/drive/MyDrive/odonto-ai/models/last.pt
